# Compare VLG-CBM backbones

Standalone companion to `evaluate_vlgcbm_colab_portable.ipynb` -- same setup, but every
model at once instead of one. Split out because `evaluate_many` loads each backbone in
turn and is much slower than the single-model sections.

Nothing is hand-carried: code from GitHub, images from a CC0 HF mirror, annotations and
weights from a GitHub Release, backbones from public hubs.

Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
!pip install -q open_clip_torch ftfy regex loguru "setuptools<81"

In [ ]:
import os, subprocess
REPO_DIR = "/content/VLG-CBM"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", "bioclip-birds525",
                    "https://github.com/aygovind/VLG-CBM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo:", os.getcwd())

## Annotations + trained weights

From the GitHub Release -- ~60MB, no auth.

In [ ]:
import subprocess, os

REL = "https://github.com/aygovind/VLG-CBM/releases/download/birds525-eval-v1"

for name, dest in [("annotations-val.tar.gz", "."), ("models-birds525.tar.gz", ".")]:
    if not os.path.exists(name):
        subprocess.run(["wget", "-q", "--show-progress", f"{REL}/{name}"], check=True)
    subprocess.run(["tar", "xzf", name, "-C", dest], check=True)

print("annotations:", len(os.listdir("annotations/birds525_val")))
print("models:", sorted(os.listdir("saved_models")))

## Images

The original Kaggle dataset was removed (404), so these come from a CC0 HF mirror.
Verified by image content and ordering against the manifest -- annotations are keyed
by ImageFolder position, so a different dataset revision would silently misalign
every box rather than error.

In [ ]:
import os, json, shutil, subprocess, hashlib

# The original Kaggle dataset (gpiosenka/100-bird-species) now 404s -- it was removed.
# This HF mirror is CC0, needs no auth, and its valid split was verified byte-identical
# to the images the annotations were generated against.
MIRROR = ("https://huggingface.co/datasets/chriamue/bird-species-dataset"
          "/resolve/main/data/valid.tar.gz")

if not os.path.isdir("datasets/birds525/val"):
    if not os.path.exists("valid.tar.gz"):
        subprocess.run(["wget", "-q", "--show-progress", "-O", "valid.tar.gz", MIRROR], check=True)
    subprocess.run(["tar", "xzf", "valid.tar.gz", "-C", "/content"], check=True)
    os.makedirs("datasets/birds525", exist_ok=True)
    if os.path.exists("datasets/birds525/val"):
        shutil.rmtree("datasets/birds525/val")
    shutil.move("/content/valid", "datasets/birds525/val")

# Verify by image CONTENT and order, not filename. Annotations are keyed by ImageFolder
# position, so what matters is that the images and their ordering match -- the mirror
# names files 1.jpg where the Kaggle original used 00001.jpg, so a path-based check
# would spuriously fail while a content check catches an actually-wrong dataset.
from torchvision import datasets
man = json.load(open("concept_files/birds525_val_manifest.json"))
ds = datasets.ImageFolder("datasets/birds525/val")
digests = [hashlib.sha256(open(p, "rb").read()).hexdigest() for p, _ in ds.samples]
got = hashlib.sha256("".join(digests).encode()).hexdigest()

if got != man["content_order_sha256"]:
    raise AssertionError(
        f"val images do not match the annotations.\n"
        f"  expected {man['n_images']} images / {man['n_classes']} classes "
        f"(content-order {man['content_order_sha256'][:12]})\n"
        f"  got      {len(digests)} images / {len(ds.classes)} classes "
        f"(content-order {got[:12]})\n"
        "Annotations are keyed by position, so the box overlays would be misaligned.")
print(f"images verified: {len(digests)} across {len(ds.classes)} classes, content-order matches")

In [ ]:
os.environ["DATASET_FOLDER"] = "/content/VLG-CBM/datasets"

import torch
import vlgcbm_analysis as va

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("models:", ", ".join(va.list_models("birds525")))

## Load every model

`evaluate_many` releases each backbone before loading the next, so they are not all
GPU-resident at once. Expect a few minutes per model on a cold cache; the per-model
`eval_val.pt` caches shipped in the release make the prediction pass itself instant.

In [ ]:
outs = va.evaluate_many(va.find_runs(), split=SPLIT, keep_concept_acts=True)
for o in outs:
    print(f"{o.name:28s} acc {o.accuracy * 100:6.2f}%")

## Accuracy and sparsity

Accuracy alone is not comparable across these: VLG-CBM's own argument is that it only
means something at a fixed number of effective concepts. `concepts_per_class` is what
makes the accuracy column readable -- and for the paper's ANEC metric proper, use
`sparse_evaluation.py`.

In [ ]:
va.compare(va.find_runs(), split=SPLIT)

## Qualitative grid

One row per model. Green = correct and the shown concepts hold up; amber = correct but
the shown concepts do not by themselves account for it (a proxy, not a verdict).

In [ ]:
va.story_figure(outs, split=SPLIT, top_concepts=2,
                flag_mode="sufficiency",
                save_path="figures/qualitative_comparison.png")